# Tennis Player Actions EDA - Hanyi

Within this notebook, we aim to conduct exploratory data analysis on the Tennis Players Actions dataset from Kaggle, available at: [Tennis Player Actions Dataset](https://www.kaggle.com/datasets/orvile/tennis-player-actions-dataset/data).

## Setup
Required dependencies:

In [1]:
!pip3 install kagglehub

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


Move to root directory:

In [5]:
import os

if os.path.basename(os.getcwd()) == "eda":
    os.chdir("..")

print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection


We begin by downloading the dataset from kagglehub:

In [ ]:
import kagglehub
import os

# Set download path to root dir, kagglehub automatically creates datasets subdir
os.environ["KAGGLEHUB_CACHE"] = ""

# Download latest version
path = kagglehub.dataset_download("orvile/tennis-player-actions-dataset")

print("Path to dataset files:", path)

100%|██████████| 503M/503M [00:14<00:00, 35.2MB/s] 

Extracting files...


Path to dataset files: datasets/orvile/tennis-player-actions-dataset/versions/1


Now, we load the dataset into a pandas df:

In [19]:
import json
import pandas as pd

def init_df(pose: str, show_info: bool = True):
    with open(f"{path}/Tennis Player Actions Dataset for Human Pose Estimation/annotations/{pose}.json") as f:
        pose_data = json.load(f)
    data_dict = {}
    data_dict["images"] = pd.DataFrame(pose_data["images"])
    data_dict["categories"] = pd.DataFrame(pose_data["categories"])
    data_dict["annotations"] = pd.DataFrame(pose_data["annotations"])

    if show_info:
        print(f"\n{'='*50}")
        print(f"Pose: {pose}")
        print(f"Images: {len(data_dict['images'])}")
        print(f"Categories: {len(data_dict['categories'])}")
        print(f"Annotations: {len(data_dict['annotations'])}\n")
        print(f"Images head:\n{data_dict['images'].head(1)}\n")
        print(f"Categories:\n{data_dict['categories']}\n")
        print(f"Annotations head:\n{data_dict['annotations'].head(1)}\n")
        print(f"{'='*50}\n")

    return data_dict

backhand_data = init_df("backhand")
forehand_data = init_df("forehand", show_info=False)
ready_position_data = init_df("ready_position", show_info=False)
serve_data = init_df("serve", show_info=False)





Pose: backhand
Images: 500
Categories: 1
Annotations: 500

Images head:
      id  dataset_id                           path  width  height   file_name
0  10001          10  ../images/backhand/B_001.jpeg   1280     720  B_001.jpeg

Categories:
   id      name                                          keypoints  \
0   5  Backhand  [nose, left_eye, right_eye, left_ear, right_ea...   

                                            skeleton  
0  [[1, 2], [1, 3], [1, 18], [2, 4], [3, 5], [6, ...  

Annotations head:
     id  image_id  category_id  \
0  3501     10001            5   

                                        segmentation   area  \
0  [[667.9, 257.3, 667.9, 467.4, 496.4, 467.4, 49...  36120   

                           bbox  iscrowd  isbbox  \
0  [496.0, 257.0, 172.0, 210.0]    False    True   

                                           keypoints  num_keypoints color  
0  [601, 291, 1, 596, 288, 1, 605, 286, 1, 592, 2...             18   NaN  




From importing the dataset, we see it is split into four classes:
1. Backhand
2. Forehand
3. Ready position
4. Serve

Looking at the annotation df, we note the important fields are "segmentation", "bbox", and "keypoints":
- Segmentation is the image segmentation mask of the player, which is a polygon that shows where in the image the player is.
- Bbox is the bounding box of the player, a rectangular box that encloses the player.
- Keypoints are the 18 keypoints that outline the specific vertices of a player, such as their head, hand, or feet. In COCO, keypoints are the vertices, while the skeletons are the edges that connects the keypoints.

In [20]:
backhand_data["annotations"].head(1)

,id,image_id,category_id,segmentation,area,bbox,iscrowd,isbbox,keypoints,num_keypoints,color
0,3501,10001,5,"[[667.9, 257.3, 667.9, 467.4, 496.4, 467.4, 49...",36120,"[496.0, 257.0, 172.0, 210.0]",False,True,"[601, 291, 1, 596, 288, 1, 605, 286, 1, 592, 2...",18,NaN


## Numeric Columns
First, we'll analyze the numeric columns:

In [29]:
backhand_data["annotations"].select_dtypes(include="number").agg(['min', 'max'])

,id,image_id,category_id,area,num_keypoints
min,3501,10001,5,0,16
max,4000,10500,5,142881,18


In [23]:
forehand_data["annotations"].select_dtypes(include="number").agg(['min', 'max'])

,id,image_id,category_id,area,num_keypoints
min,4001,10501,6,13980,17
max,5502,11000,6,145803,18


In [24]:
ready_position_data["annotations"].select_dtypes(include="number").agg(['min', 'max'])

,id,image_id,category_id,area,num_keypoints
min,4500,11001,7,11773,17
max,4999,11500,7,120218,18


In [25]:
serve_data["annotations"].select_dtypes(include="number").agg(['min', 'max'])

,id,image_id,category_id,area,num_keypoints
min,5000,11501,8,0,17
max,5501,12000,8,66123,18


We note that the numeric fields are:
- `id`: The annotation ID of the current sample. Maps 1:1 with `image_id`. Ranges from 3501 to 5501, where each class is sorted in ascending order.
- `image_id`: The image ID of the current sample. Same attributes as `id`, but ranges from 10001 to 12000.
- `category_id`: The class ID of each sample. 5 = backhand, 6 = forehand, 7 = ready position, 8 = serve.
- `area`: The area of the segmentation mask, NOT the bbox.
- `num_keypoints`: The number of vertices for the pose skeleton of the current sample. There are 18 defined vertices, but some samples may have up to 2 missing.

Each annotation is a dictionary with the following keys:

- `id`: 
- `image_id`:
- `category_id`:
- `segmentation`:
- `area`:
- `bbox`:
- `iscrowd`:
- `isbbox`:
- `keypoints`:
- `num_keypoints`:
- `color`: